In [1]:
import pandas as pd
import numpy as np
import os
import glob

# ── Configuration ────────────────────────────────────────────────────────────
ROOT = r"D:\FCAI\plain_coupon\infared_data"
PIXEL_SIZE = 0.045 / 150          # 0.0003 m per pixel

# Window origin (x-coordinate of index 0) and direction
# Both windows: index ordering is in the REVERSE streamwise direction
# so  x = origin - index * pixel_size
WINDOW_ORIGINS = {
    "window_1_spanwise_avg_std.csv": 0.1675,   # m
    "window_2_spanwise_avg_std.csv": 0.1000,   # m
}

SUMMARY_SUBPATH = os.path.join(
    "time_average", "average_temperature",
    "half_domain_temperature", "summary_avg_std"
)

OUTPUT_FILENAME = "combined_spanwise_avg_std.csv"

# ── Find all case directories ─────────────────────────────────────────────────
case_dirs = [
    d for d in os.listdir(ROOT)
    if os.path.isdir(os.path.join(ROOT, d)) and d.startswith("cr_")
]

print(f"Found {len(case_dirs)} case(s): {case_dirs}\n")

for case in case_dirs:
    summary_dir = os.path.join(ROOT, case, SUMMARY_SUBPATH)

    if not os.path.isdir(summary_dir):
        print(f"  [SKIP] {case}: summary directory not found → {summary_dir}")
        continue

    dfs = []
    for fname, origin in WINDOW_ORIGINS.items():
        fpath = os.path.join(summary_dir, fname)
        if not os.path.isfile(fpath):
            print(f"  [WARN] {case}: file not found → {fpath}")
            continue

        df = pd.read_csv(fpath)
        df.columns = ["col_index", "spanwise_avg_C", "spanwise_std_C"]

        # Calibrate: world coordinate [m], reverse streamwise
        df["x_m"] = origin - df["col_index"].astype(float) * PIXEL_SIZE
        dfs.append(df[["x_m", "spanwise_avg_C", "spanwise_std_C"]])

    if not dfs:
        print(f"  [SKIP] {case}: no data files loaded.")
        continue

    combined = pd.concat(dfs, ignore_index=True)
    combined.sort_values("x_m", inplace=True)
    combined.reset_index(drop=True, inplace=True)
    combined.columns = ["x_world_m", "spanwise_avg_C", "spanwise_std_C"]

    out_path = os.path.join(summary_dir, OUTPUT_FILENAME)
    combined.to_csv(out_path, index=False)
    print(f"  [OK] {case}: wrote {len(combined)} rows → {out_path}")

print("\nDone.")

Found 15 case(s): ['cr_1400_phi_085', 'cr_1400_phi_09', 'cr_1400_phi_10', 'cr_1400_phi_118', 'cr_1400_phi_137', 'cr_2300_phi_085', 'cr_2300_phi_09', 'cr_2300_phi_10', 'cr_2300_phi_118', 'cr_2300_phi_137', 'cr_500_phi_085', 'cr_500_phi_09', 'cr_500_phi_10', 'cr_500_phi_118', 'cr_500_phi_137']

  [OK] cr_1400_phi_085: wrote 300 rows → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_085\time_average\average_temperature\half_domain_temperature\summary_avg_std\combined_spanwise_avg_std.csv
  [OK] cr_1400_phi_09: wrote 300 rows → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_09\time_average\average_temperature\half_domain_temperature\summary_avg_std\combined_spanwise_avg_std.csv
  [OK] cr_1400_phi_10: wrote 300 rows → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_10\time_average\average_temperature\half_domain_temperature\summary_avg_std\combined_spanwise_avg_std.csv
  [OK] cr_1400_phi_118: wrote 300 rows → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_118\time_average\average_temperature\half